# Laboratorio 4 - Segmentación de imágenes con técnicas de agrupación

*Presentado por:*
- Shaiel Jiménez - 202323846
- Juan Esteban Triviño - 202315338

*ISIS2611 - Aprendizaje de Máquina / Sección 5*

*Profesor: Rubén Manrique*

## <a id="entorno"></a> Configuración del entorno

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, v_measure_score, f1_score, precision_score, recall_score

SEMILLA = 42

print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

### Datos

Se usan 20 imágenes dermatoscópicas del conjunto ISIC 2016, tarea de segmentación de lesiones ([enlace](https://challenge.isic-archive.com/landing/2016/37/)), cada una con su máscara de referencia elaborada por especialistas. Siguiendo el enunciado, 15 imágenes son de ajuste y 5 de prueba. Las de prueba y sus máscaras no se abren hasta la Actividad 5, así que aquí solo se comprueba que los archivos existen.

Los datos van en *data/raw/ajuste* y *data/raw/prueba*, con los nombres originales (*ISIC_0000085.jpg* y *ISIC_0000085_Segmentation.png*). Las rutas se arman con *pathlib* y son relativas a la carpeta del notebook, para que funcionen igual en macOS, Linux o Windows.

In [ ]:
DATA_DIR = Path("data") / "raw"

IDS_AJUSTE = ["ISIC_0000085", "ISIC_0000086", "ISIC_0000089", "ISIC_0000094", "ISIC_0000108",
              "ISIC_0000114", "ISIC_0000116", "ISIC_0000122", "ISIC_0000133", "ISIC_0000137",
              "ISIC_0000150", "ISIC_0000153", "ISIC_0000156", "ISIC_0000157", "ISIC_0000112"]
IDS_PRUEBA = ["ISIC_0000087", "ISIC_0000105", "ISIC_0000110", "ISIC_0000123", "ISIC_0000128"]

def carpeta(id_img):
    return DATA_DIR / ("prueba" if id_img in IDS_PRUEBA else "ajuste")

def ruta_imagen(id_img):
    return carpeta(id_img) / f"{id_img}.jpg"

def ruta_mascara(id_img):
    return carpeta(id_img) / f"{id_img}_Segmentation.png"

# Solo se verifica que existan los archivos, sin abrirlos
todos = IDS_AJUSTE + IDS_PRUEBA
faltantes = [str(r) for i in todos for r in (ruta_imagen(i), ruta_mascara(i)) if not r.exists()]
print(f"IDs: {len(IDS_AJUSTE)} de ajuste + {len(IDS_PRUEBA)} de prueba = {len(set(todos))}")
print(f"Archivos esperados: {2 * len(todos)} | faltantes: {len(faltantes)} {faltantes if faltantes else ''}")
print("IDs en ajuste y prueba a la vez:", set(IDS_AJUSTE) & set(IDS_PRUEBA))

## <a id="exploracion"></a> Exploración de las imágenes de ajuste

### Carga y verificación

Se cargan las 15 imágenes de ajuste con sus máscaras y se revisa lo que el enunciado da por hecho: que cada máscara tiene el mismo tamaño que su imagen y solo los valores 0 (piel o fondo) y 255 (lesión).

In [ ]:
def cargar(id_img):
    img = np.array(Image.open(ruta_imagen(id_img)).convert("RGB"))
    mascara = np.array(Image.open(ruta_mascara(id_img)))
    return img, mascara

originales = {i: cargar(i) for i in IDS_AJUSTE}

filas = []
for id_img, (img, mascara) in originales.items():
    assert img.shape[:2] == mascara.shape, f"{id_img}: tamaños distintos"
    filas.append({"id": id_img, "alto": img.shape[0], "ancho": img.shape[1],
                  "ancho/alto": round(img.shape[1] / img.shape[0], 2),
                  "valores máscara": sorted(np.unique(mascara).tolist()),
                  "% lesión": round((mascara > 127).mean() * 100, 1)})
resumen_ajuste = pd.DataFrame(filas).set_index("id")
resumen_ajuste